In [3]:
import numpy as np
import pandas as pd
import random
import os
from math import ceil
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from itertools import combinations
from sklearn.mixture import GaussianMixture

from copulas.multivariate import VineCopula
from copulas.univariate import GaussianKDE

output_folder_cluster = '/home/jupyter-iec_23se07/GMM-nfst/Datascaled/OutlierData'  # Folder for cluster data

os.makedirs(output_folder_cluster, exist_ok=True)

import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import gaussian_kde
from copulas.multivariate import VineCopula
import pandas as pd

def generate_realistic_synthetic(X, y, realistic_synthetic_mode, alpha:int, percentage:float, seed:int=42):
    '''
    Currently, four types of realistic synthetic outliers can be generated:
    1. local outliers: where normal data follows the GMM distribution, and anomalies follow the GMM distribution with modified covariance
    2. global outliers: where normal data follows the GMM distribution, and anomalies follow the uniform distribution
    3. dependency outliers: where normal data follows the vine copula distribution, and anomalies follow the independent distribution captured by GaussianKDE
    4. cluster outliers: where normal data follows the GMM distribution, and anomalies follow the GMM distribution with modified mean

    :param X: input X
    :param y: input y
    :param realistic_synthetic_mode: the type of generated outliers
    :param alpha: the scaling parameter for controlling the generated local and cluster anomalies
    :param percentage: controlling the generated global anomalies
    :param seed: random seed for reproducibility
    '''

    if realistic_synthetic_mode not in ['local', 'cluster', 'dependency', 'global']:
        raise NotImplementedError(f"Mode {realistic_synthetic_mode} is not implemented.")

    # the number of normal data and anomalies
    pts_n = len(np.where(y == 0)[0])
    pts_a = len(np.where(y == 1)[0])

    
    # only use the normal data to fit the model
    X_normal = X[y.values == 0]
    y_normal = y[y == 0]

    # generate the synthetic normal data
    if realistic_synthetic_mode in ['local', 'cluster', 'global']:
        # select the best n_components based on the BIC value
        metric_list = []
        n_components_list = list(np.arange(1, 10))

        for n_components in n_components_list:
            gm = GaussianMixture(n_components=n_components, random_state=seed).fit(X_normal)
            metric_list.append(gm.bic(X_normal))

        best_n_components = n_components_list[np.argmin(metric_list)]

        # refit based on the best n_components
        gm = GaussianMixture(n_components=best_n_components, random_state=seed).fit(X_normal)

        # generate the synthetic normal data
        X_synthetic_normal = gm.sample(pts_n)[0]

    elif realistic_synthetic_mode == 'dependency':
        # sampling the feature since copulas method may spend too long to fit
        if X.shape[1] > 50:
            idx = np.random.choice(np.arange(X.shape[1]), 50, replace=False)
            X_normal = X_normal.iloc[:, idx]
        
        copula = VineCopula('center')  # default is the C-vine copula
        if X_normal.shape[0] > 2000:
            X_sampled = X_normal.sample(n=2000, random_state=42)  # Sample 2000 rows
        else:
            X_sampled = X_normal  # Use all rows if there are fewer than 2000
        
        print( " TOI DAY ROI " * 5)
        
        copula.fit(pd.DataFrame(X_sampled))

        
        # sample to generate synthetic normal data
        X_synthetic_normal = copula.sample(pts_n).values
        print(X_synthetic_normal)
       
    else:
        pass

    # generate the synthetic abnormal data
    if realistic_synthetic_mode == 'local':
        # generate the synthetic anomalies (local outliers)
        gm.covariances_ = alpha * gm.covariances_
        X_synthetic_anomalies = gm.sample(pts_a)[0]

    elif realistic_synthetic_mode == 'cluster':
        # generate the clustering synthetic anomalies
        gm.means_ = alpha * gm.means_
        X_synthetic_anomalies = gm.sample(pts_a)[0]

    elif realistic_synthetic_mode == 'dependency':
        print("helo")
        X_synthetic_anomalies = np.zeros((pts_a, X_normal.shape[1]))
        print("helo")
        
        # using the GaussianKDE for generating independent features
        for i in range(X_normal.shape[1]):
            kde = GaussianKDE()
            kde.fit(X_normal.iloc[:, i])  # Use .iloc to index columns by position
            X_synthetic_anomalies[:, i] = kde.sample(pts_a)
        print("helo")
    elif realistic_synthetic_mode == 'global':
        # generate the synthetic anomalies (global outliers)
        X_synthetic_anomalies = []

        for i in range(X_synthetic_normal.shape[1]):
            low = np.min(X_synthetic_normal[:, i]) * (1 + percentage)
            high = np.max(X_synthetic_normal[:, i]) * (1 + percentage)

            X_synthetic_anomalies.append(np.random.uniform(low=low, high=high, size=pts_a))

        X_synthetic_anomalies = np.array(X_synthetic_anomalies).T

    else:
        pass

    # Concatenate normal and anomalous data
    X_combined = np.concatenate((X_synthetic_normal, X_synthetic_anomalies), axis=0)
    y_combined = np.append(np.repeat(0, X_synthetic_normal.shape[0]),
                           np.repeat(1, X_synthetic_anomalies.shape[0]))

    return X_combined, y_combined



# Function to load and process the dataset
def load_and_process_dataset(name, scaler):
    try:
        data = pd.read_csv(f"/home/jupyter-iec_23se07/GMM-nfst/Datascaled/{scaler}_{name}")
        
        return data
    except Exception as e:
        print(f"Error loading dataset {name}: {e}")
        return None

# Dataset links

datasets = ['data_ToNIoT.csv'] # 'data_N_BaIoT.csv', 'data_CICIoT2023.csv','data_BoTIoT.csv', ]

# datasets = ['data_BoTIoT.csv']

scaler_names = ['MinMaxScaler', 'Normalizer', 'StandardScaler', 'QuantileTransformer', 'RobustScaler']

# scaler_names = ['StandardScaler','QuantileTransformer','MinMaxScaler','Normalizer']


outlier_modes = {
    # 'dependency' : 0 #, 
    'local' : 5,
    'cluster' : 5, 
    'global' : 1.1, 
}


for (outlier_mode, alpha) in outlier_modes.items() : 
    
    for scaler in scaler_names: 
        
        for dataset in datasets:
                print( outlier_mode, scaler, dataset ) 
                data = load_and_process_dataset(dataset, scaler)
                if data is None:
                    continue  # Skip if dataset couldn't be loaded
        
                try:
                    X = data.iloc[:, :-1].to_numpy()
                    y = data.iloc[:, -1].to_numpy()
                     
                    X, y = pd.DataFrame(X), pd.DataFrame(y)
        
                    print("Original data size:", len(y))
        
                
                    # Reduce data size if too large
                    if len(y) > 10000:
                        print("Reducing data size to 10000")
                        _, X, _, y = train_test_split(X, y, test_size=10000, random_state=42)
            
                    
                    X_cluster, y_cluster = generate_realistic_synthetic(X, y, outlier_mode, alpha, percentage=0.1)
                    # Convert the synthetic data to DataFrame before saving
    
                    X_cluster_df = pd.DataFrame(X_cluster)
                    y_cluster_df = pd.DataFrame(y_cluster, columns=["label"])  # Đặt tên cột cho nhãn
                    
                    # Nối y_cluster vào X_cluster theo chiều cột (axis=1)
                    cluster_df = pd.concat([X_cluster_df, y_cluster_df], axis=1)
    
                    train_data, test_data = train_test_split(cluster_df, test_size=0.3)
                    print( train_data.shape, test_data.shape) 
                    # Save synthetic data for "cluster"

                    
                    train_file_path = os.path.join(output_folder_cluster, f'Test_{outlier_mode}_{scaler}_{dataset}')
                    test_file_path = os.path.join(output_folder_cluster, f'Train_{outlier_mode}_{scaler}_{dataset}')
        
                    train_data.to_csv(train_file_path, index=False)
                    test_data.to_csv(test_file_path, index=False)
    
                    print(f"Saved synthetic data (cluster) for {dataset} in folder '{output_folder_cluster}'")
                except Exception as e:
                    print(f"Error processing dataset {dataset}: {e}")
                


local MinMaxScaler data_ToNIoT.csv
Original data size: 48785
Reducing data size to 10000
(7000, 29) (3000, 29)
Saved synthetic data (cluster) for data_ToNIoT.csv in folder '/home/jupyter-iec_23se07/GMM-nfst/Datascaled/OutlierData'
local Normalizer data_ToNIoT.csv
Original data size: 48785
Reducing data size to 10000
(7000, 29) (3000, 29)
Saved synthetic data (cluster) for data_ToNIoT.csv in folder '/home/jupyter-iec_23se07/GMM-nfst/Datascaled/OutlierData'
local StandardScaler data_ToNIoT.csv
Original data size: 48785
Reducing data size to 10000
(7000, 29) (3000, 29)
Saved synthetic data (cluster) for data_ToNIoT.csv in folder '/home/jupyter-iec_23se07/GMM-nfst/Datascaled/OutlierData'
local QuantileTransformer data_ToNIoT.csv
Original data size: 48785
Reducing data size to 10000
(7000, 29) (3000, 29)
Saved synthetic data (cluster) for data_ToNIoT.csv in folder '/home/jupyter-iec_23se07/GMM-nfst/Datascaled/OutlierData'
local RobustScaler data_ToNIoT.csv
Original data size: 48785
Reducing